# MoMA Art Mining — Midterm Progress Report
**Ahmed Sahigara | Manipal Institute of Technology**

---

## Objective
Apply data mining and ML techniques to the [MoMA collection dataset](https://github.com/MuseumofModernArt/collection) to:
- Automatically classify artworks into departments/categories
- Discover hidden groupings via unsupervised clustering

This notebook covers **Phase 1**: dataset exploration, cleaning, feature engineering, and baseline models.

In [ ]:
import sys
sys.path.insert(0, '..')   # repo root

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import ensure_dirs
ensure_dirs()

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

---
## 1. Dataset Overview

In [ ]:
from src.data_loader import load_raw, merge_datasets

artworks_raw, artists_raw = load_raw()
print('\n--- Artworks (first 3 rows) ---')
artworks_raw.head(3)

In [ ]:
print('--- Artists (first 3 rows) ---')
artists_raw.head(3)

In [ ]:
# Column types and null counts at a glance
print('Artworks info:')
artworks_raw.info()
print('\nArtists info:')
artists_raw.info()

---
## 2. Data Cleaning

In [ ]:
from src.data_loader import load_and_clean

df = load_and_clean()
df.head()

In [ ]:
# Check what we engineered during cleaning
new_cols = ['creation_year', 'creation_decade', 'acquisition_year',
            'years_to_acquire', 'height_cm', 'width_cm', 'area_cm2', 'is_female']
df[[c for c in new_cols if c in df.columns]].describe()

---
## 3. Exploratory Data Analysis

In [ ]:
from src.eda import department_distribution
fig = department_distribution(df)
plt.show()

In [ ]:
from src.eda import missing_values_heatmap
fig = missing_values_heatmap(df)
plt.show()

In [ ]:
from src.eda import timeline_plot
fig = timeline_plot(df)
plt.show()

In [ ]:
from src.eda import nationality_chart, gender_split
fig = nationality_chart(df)
plt.show()

fig = gender_split(df)
plt.show()

In [ ]:
from src.eda import medium_top20
fig = medium_top20(df)
plt.show()

In [ ]:
from src.eda import acquisition_trend
fig = acquisition_trend(df)
plt.show()

### Key EDA Findings
- **Prints & Illustrated Books** and **Photography** dominate the collection by volume.
- Most artworks were created post-1950, reflecting MoMA's modern/contemporary focus.
- Artist nationalities are heavily skewed toward American and European artists.
- ~85% of represented artists are male — a well-known bias in major museum collections.
- The Medium column is rich but messy — TF-IDF will handle it well.

---
## 4. Feature Engineering

In [ ]:
from src.features import extract_features

clf_df, clust_df, meta = extract_features(df)

print('\nClassification feature matrix:')
print(f'  Shape : {clf_df.shape}')
print(f'  Classes : {meta["class_names"]}')
clf_df.head(3)

In [ ]:
# Show which features are available (split by type)
feat_names = meta['feature_names']
structured = [f for f in feat_names if not f.startswith('med_')]
tfidf      = [f for f in feat_names if f.startswith('med_')]

print(f'Structured features ({len(structured)}): {structured}')
print(f'\nTF-IDF features ({len(tfidf)}): {tfidf[:10]}...')

In [ ]:
from src.eda import correlation_matrix
fig = correlation_matrix(clf_df)
plt.show()

---
## 5. Baseline Classification Model

In [ ]:
from src.classifier import prepare_data, train_logistic, evaluate

X_train, X_test, y_train, y_test, scaler, feat_cols = prepare_data(clf_df)

In [ ]:
print('Training Logistic Regression baseline...')
lr = train_logistic(X_train, y_train)
lr_result = evaluate(lr, X_test, y_test, meta['class_names'], 'Logistic Regression')
plt.show()

In [ ]:
# Cross-validation to check generalization
from src.classifier import cross_validate_model
cv_scores = cross_validate_model(lr, X_train, y_train, 'Logistic Regression')

---
## 6. Preliminary Clustering

In [ ]:
from sklearn.preprocessing import StandardScaler
from src.clusterer import reduce_pca, find_optimal_k

scaler_clust = StandardScaler()
X_scaled = scaler_clust.fit_transform(clust_df.values)

X_pca50, _ = reduce_pca(X_scaled, n_components=min(50, X_scaled.shape[1]))
X_2d, _    = reduce_pca(X_scaled, n_components=2)

In [ ]:
from src.visualizer import pca_explained_variance
fig = pca_explained_variance(X_scaled)
plt.show()

In [ ]:
best_k, fig = find_optimal_k(X_pca50)
plt.show()
print(f'\nSuggested k = {best_k}')

In [ ]:
from src.clusterer import run_kmeans, plot_pca_clusters

km_labels, km_model = run_kmeans(X_pca50, best_k)
fig = plot_pca_clusters(X_2d, km_labels, title=f'PCA — KMeans k={best_k} (midterm preview)')
plt.show()

---
## 7. Summary & Next Steps

**Done:**
- [X] Dataset loaded and cleaned (~130K artworks after filtering)
- [X] 8 EDA visualisations reveal collection structure and biases
- [X] Feature matrix: structured metadata + 100 TF-IDF medium features
- [X] Logistic Regression baseline with cross-validation
- [X] KMeans preliminary clusters (elbow + silhouette)

**To-do:**
- [ ] Random Forest + SVM, with model comparison
- [ ] Feature importance analysis
- [ ] DBSCAN clustering
- [ ] t-SNE visualisation
- [ ] Cluster interpretation (what do the groups *mean* artistically?)
- [ ] Summary dashboard